# 머신러닝 기반 텍스트 분류
1. 데이터 준비 : 로딩, 입력데이터, 출력데이터, 학습/테스트 데이터 분리, 특징추출
2. 학습-평가
3. 배포 준비

### 1. 데이터 준비

In [1]:
import pandas as pd

datafile = './data/Korean_movie_reviews_2016.csv'
data_df = pd.read_csv(datafile)
data_df.head()

,review,label
0,부산 행 때문 너무 기대하고 봤,0
1,한국 좀비 영화 어색하지 않게 만들어졌 놀랍,1
2,조금 전 보고 왔 지루하다 언제 끝나 이 생각 드,0
3,평 밥 끼 먹자 돈 니 내고 미친 놈 정신사 좀 알 싶어 그래 밥 먹다 먹던 숟가락...,1
4,점수 대가 과 이 엑소 팬 어중간 점수 줄리 없겠 클레멘타인 이후 최고 평점 조작 ...,0


In [2]:
data_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 165384 entries, 0 to 165383
Data columns (total 2 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   review  165384 non-null  object
 1   label   165384 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 2.5+ MB


In [3]:
# 입력 데이터, 정답 데이터 추출
review_list = list(data_df.review)    # 입력 데이터
label_list = list(data_df.label)      # 출력 데이터
len(review_list), len(label_list)

(165384, 165384)

In [4]:
# 학습 데이터, 테스트 데이터 분리
from sklearn.model_selection import train_test_split

train_X, test_X, train_y, test_y = train_test_split(review_list, label_list, test_size=0.1)
len(train_X), len(test_X), len(train_y), len(test_y)

(148845, 16539, 148845, 16539)

In [5]:
# 한국어 토크나이저 정의
from konlpy.tag import Okt
def korean_tokenizer(text):
    my_tags = ['Noun', 'Adjective', 'Verb']
    my_stopwords = ["내", "내내", "티", "나", "들인건", "할수밖에", "없다", "보는", "정말", "하는", "보고", 
                    "입니다", "그냥", "정도", "해서", "있는", "봤는데", "거", "것", "그", 
                    "꼭", "더", "듯", 
                    "때", "뭐", "볼", "수", "안", "왜", 
                    "이", "점", "좀", "편"]
    tokenizer = Okt().pos
    return [word for word,  tag in tokenizer(text) if tag in my_tags and word not in my_stopwords]


In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(tokenizer=korean_tokenizer, max_features=1000)
vectorizer.fit(train_X)

c:\Users\user\anaconda3\envs\textmine26\lib\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


TfidfVectorizer(max_features=1000,
                tokenizer=<function korean_tokenizer at 0x000001B5526C3B80>)

In [7]:
len(vectorizer.get_feature_names_out()), vectorizer.get_feature_names_out()[:10]

(1000,
 array(['가', '가고', '가는', '가볍', '가서', '가슴', '가장', '가족', '가지', '각본'],
       dtype=object))

In [8]:
# 학습 데이터 특징 추출
train_X_fv = vectorizer.transform(train_X)

In [9]:
# 테스트 데이터 특징 추출
test_X_fv = vectorizer.transform(test_X)

In [10]:
print(train_X_fv)

  (np.int32(0), np.int32(103))	0.5058149050192656
  (np.int32(0), np.int32(210))	0.3492206203375851
  (np.int32(0), np.int32(312))	0.2614449194145246
  (np.int32(0), np.int32(369))	0.27106473606676285
  (np.int32(0), np.int32(387))	0.23031539135295404
  (np.int32(0), np.int32(418))	0.23182350592360454
  (np.int32(0), np.int32(477))	0.38855694611669117
  (np.int32(0), np.int32(573))	0.3867924114861243
  (np.int32(0), np.int32(771))	0.27017497258853573
  (np.int32(1), np.int32(210))	0.6323246653776801
  (np.int32(1), np.int32(278))	0.5406712291020475
  (np.int32(1), np.int32(279))	0.5548334340829564
  (np.int32(2), np.int32(439))	0.5332752908170868
  (np.int32(2), np.int32(626))	0.3261593411703408
  (np.int32(2), np.int32(657))	0.43072640385719824
  (np.int32(2), np.int32(738))	0.4290816777321509
  (np.int32(2), np.int32(926))	0.48949078359667725
  (np.int32(3), np.int32(232))	0.6975032978435329
  (np.int32(3), np.int32(386))	0.30840406324204594
  (np.int32(3), np.int32(387))	0.249393587

In [11]:
# 정답 데이터를 ndarray 변환 np.array(sequence)
import numpy as np
train_y = np.array(train_y)
test_y = np.array(test_y)
print(train_y[:10])

[1 0 0 0 1 0 0 1 0 0]


# 2. 머신러닝 - 모델 학습
1. 의사결정트리, Decision Tree (DT)
2. 랜덤포레스트, Random Forest (RF)

In [12]:
# 모델별 정확도를 dataframe으로 저장하여 비교
import pandas as pd
score_df = pd.DataFrame(columns=['train', 'test'])
score_df

,train,test


In [13]:
def get_scores(model, train_X, train_y, test_X, test_y):
    train_score = model.score(train_X, train_y) * 100
    test_score = model.score(test_X, test_y) * 100
    return train_score, test_score

### 1. Decision Tree

In [14]:
from sklearn.tree import DecisionTreeClassifier

dtc = DecisionTreeClassifier()
dtc.fit(train_X_fv, train_y)

DecisionTreeClassifier()

In [15]:
train_score, test_score = get_scores(dtc, train_X_fv, train_y, test_X_fv, test_y)
print(train_score, test_score)

97.46380462897645 80.24064332789165


In [16]:
score_df.loc['DecisionTree'] = [train_score, test_score]
score_df

,train,test
DecisionTree,97.463805,80.240643


### 2. Random Forest

In [17]:
from sklearn.ensemble import RandomForestClassifier

rfc = RandomForestClassifier(n_jobs=-1)
rfc.fit(train_X_fv, train_y)

RandomForestClassifier(n_jobs=-1)

In [18]:
train_score, test_score = get_scores(rfc, train_X_fv, train_y, test_X_fv, test_y)
print(train_score, test_score)
score_df.loc['RandomForest'] = [train_score, test_score]
score_df

97.46313278914307 84.0437753189431


,train,test
DecisionTree,97.463805,80.240643
RandomForest,97.463133,84.043775


### 3. 나이브 베이즈 분류

In [19]:
from sklearn.naive_bayes import MultinomialNB

mnb = MultinomialNB()
mnb.fit(train_X_fv, train_y)

MultinomialNB()

In [20]:
train_score, test_score = get_scores(mnb, train_X_fv, train_y, test_X_fv, test_y)
print(train_score, test_score)
score_df.loc['NaiveBayes'] = [train_score, test_score]
score_df

84.82112264436158 84.42469314952537


,train,test
DecisionTree,97.463805,80.240643
RandomForest,97.463133,84.043775
NaiveBayes,84.821123,84.424693


### 4. 로지스틱 회귀 분석, logistic regression

In [21]:
from sklearn.linear_model import LogisticRegression
lr = LogisticRegression(solver='liblinear')
lr.fit(train_X_fv, train_y)

LogisticRegression(solver='liblinear')

In [22]:
train_score, test_score = get_scores(lr, train_X_fv, train_y, test_X_fv, test_y)
print(train_score, test_score)
score_df.loc['LogisticRegression'] = [train_score, test_score]
score_df

85.62867412408882 85.1865288106899


,train,test
DecisionTree,97.463805,80.240643
RandomForest,97.463133,84.043775
NaiveBayes,84.821123,84.424693
LogisticRegression,85.628674,85.186529


### 5. 서포트 벡터 머신, SVM-Support Vector Machine

In [23]:
from sklearn.svm import LinearSVC
svm = LinearSVC()
svm.fit(train_X_fv, train_y)

LinearSVC()

In [24]:
train_score, test_score = get_scores(svm, train_X_fv, train_y, test_X_fv, test_y)
print(train_score, test_score)
score_df.loc['SVM'] = [train_score, test_score]
score_df

85.59508213241963 85.1381582925207


,train,test
DecisionTree,97.463805,80.240643
RandomForest,97.463133,84.043775
NaiveBayes,84.821123,84.424693
LogisticRegression,85.628674,85.186529
SVM,85.595082,85.138158


In [25]:
score_df.sort_values(by='test', ascending=False)

,train,test
LogisticRegression,85.628674,85.186529
SVM,85.595082,85.138158
NaiveBayes,84.821123,84.424693
RandomForest,97.463133,84.043775
DecisionTree,97.463805,80.240643


### 6. 배포 준비
- 기능 구현
- 모델 저장

In [26]:
# 전처리 -> 특징 추출 -> 모델 학습
# review = '영화가 너무 재미있다'
review = '영화가 너무 재미없다'

def analyze_sentiment(review):
    # 전처리 및 특징 벡터 추출
    review_fv = vectorizer.transform([review])
    # print(review_fv)

    result = svm.predict(review_fv)
    # print(result)

    show = '긍정' if result[0] >= 0.5 else '부정'
    return show

show = analyze_sentiment(review)
print(f'{review} -> {show}')

영화가 너무 재미없다 -> 부정


In [27]:
reviews = [
    '영화가 너무 재미있다',
    '영화가 너무 재미없다',
    '개꿀잼',
    '대유잼',
    '노잼',
    '영화 보다가 졸았음'
]

for review in reviews:
    print(f'{review} -> {analyze_sentiment(review)}')

영화가 너무 재미있다 -> 긍정
영화가 너무 재미없다 -> 부정
개꿀잼 -> 긍정
대유잼 -> 긍정
노잼 -> 부정
영화 보다가 졸았음 -> 부정


In [28]:
import joblib


vectorizer_file = './model/sa_movie_vectorizer.pkl'
model_file = './model/sa_movie_model.pkl'

joblib.dump(vectorizer, vectorizer_file)
joblib.dump(svm, model_file)

['./model/sa_movie_model.pkl']